# Experiment: SOT gate-movement diagnostic (Colab)

Objective:
- Answer the question that gates every downstream decision on this repo: **does the
  Self-Organizing Transformer's gate mechanism actually move under the current training
  objective, in which direction, and where (which layers/heads/FFN groups)?**
- `architecture_cost()` at initialization is ~0.16 against the default `target_cost=0.45`,
  so the Lagrangian budget term never activates at the shipped defaults (the multiplier is
  clamped to 0 and stays there). The only thing acting on the gates is the shrink-only
  sparsity regularizer. This notebook produces the evidence needed to decide whether that's
  actually a problem, before touching any hyperparameter.
- Success criteria: a plot of gate means per layer/family over training steps that either
  shows (a) real, non-uniform divergence across layers — evidence the mechanism works and
  hyperparameters just need retuning — or (b) a uniform, near-flat shrink across every layer
  identically — evidence the regularizer dominates and nothing interesting is being selected.

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better). This will run on
CPU but will be much slower.


In [ ]:
# Setup: confirm the runtime and record it for reproducibility
import torch, platform, sys

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime > Change runtime type > T4 GPU, then re-run.")


In [ ]:
# Install the packages this repo needs that Colab doesn't ship by default.
# torch/numpy are already present in the Colab image.
!pip install -q -U transformers datasets accelerate huggingface_hub safetensors tqdm


## Plan

- Hypothesis: gate logits move measurably away from their initialization once both training
  schedules have actually finished ramping, and the movement is non-uniform across
  layers/heads/groups (not just a uniform shrink driven entirely by `arch_reg`).
- Why 2000 steps, not 500: `warmup_steps=500` means the LR schedule doesn't reach full
  strength until step 500, and `gate_warmup_steps=1000` means `arch_reg`'s own ramp doesn't
  finish until step 1000. Stopping at 500 only shows the LR-warmup transient, with the
  sparsity regularizer still at half strength. 2000 steps gives ~1000 steps of margin with
  *both* schedules fully engaged, which is the region that actually answers the question.
- Variables held fixed: model width/depth at the repo's actual Phase-1 defaults
  (`d_model=384`, `max_layers=12`, `max_heads=6`, `max_ffn_dim=1536`, `ffn_groups=64`) —
  shrinking the model for speed would answer a different question than the one that matters.
- Variable capped for speed: `--max_train_docs`, which limits how much of the ~2.1M-row
  TinyStories train split gets tokenized. Tokenizing the full split costs several minutes on
  its own, independent of GPU, for a run that only needs a few million tokens.
- Metrics recorded (already logged by `train.py` via `model.gate_summary()`): `cost` vs
  `target_cost`, `lagrange`, and per-layer means for the layer/head/ffn/skip gate families,
  every `log_every` steps, written to `outputs/diag/final/metrics.json`.


In [ ]:
%%writefile model.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Optional

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, PreTrainedModel


class SOTConfig(PretrainedConfig):
    model_type = "self_organizing_transformer"

    def __init__(
        self,
        vocab_size: int = 50257,
        max_position_embeddings: int = 512,
        max_layers: int = 12,
        d_model: int = 384,
        max_heads: int = 6,
        max_ffn_dim: int = 1536,
        ffn_groups: int = 64,
        skip_span: int = 3,
        dropout: float = 0.0,
        layer_norm_eps: float = 1e-5,
        architecture_mode: str = "sot",
        gate_init_logit: float = -1.0,
        **kwargs: Any,
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.max_position_embeddings = max_position_embeddings
        self.max_layers = max_layers
        self.d_model = d_model
        self.max_heads = max_heads
        self.max_ffn_dim = max_ffn_dim
        self.ffn_groups = ffn_groups
        self.skip_span = skip_span
        self.dropout = dropout
        self.layer_norm_eps = layer_norm_eps
        self.architecture_mode = architecture_mode
        self.gate_init_logit = gate_init_logit


class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_f = x.float()
        x_f = x_f * torch.rsqrt(x_f.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * x_f).to(dtype=x.dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, config: SOTConfig):
        super().__init__()
        assert config.d_model % config.max_heads == 0
        self.d_model = config.d_model
        self.max_heads = config.max_heads
        self.head_dim = config.d_model // config.max_heads
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
        self.out = nn.Linear(config.d_model, config.d_model, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.head_logits = nn.Parameter(torch.ones(config.max_heads) * config.gate_init_logit)
        self.register_buffer(
            "causal",
            torch.tril(torch.ones(config.max_position_embeddings, config.max_position_embeddings, dtype=torch.bool)),
            persistent=False,
        )

    def forward(self, x: torch.Tensor, head_gate: torch.Tensor) -> torch.Tensor:
        b, t, c = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.max_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self.causal[:t, :t]
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        probs = F.softmax(scores.float(), dim=-1).to(dtype=scores.dtype)
        probs = self.attn_dropout(probs)
        y = probs @ v
        y = y * head_gate.view(1, -1, 1, 1)
        y = y.transpose(1, 2).contiguous().view(b, t, c)
        return self.resid_dropout(self.out(y))


class GroupedFFN(nn.Module):
    def __init__(self, config: SOTConfig):
        super().__init__()
        assert config.max_ffn_dim % config.ffn_groups == 0
        self.max_ffn_dim = config.max_ffn_dim
        self.ffn_groups = config.ffn_groups
        self.group_size = config.max_ffn_dim // config.ffn_groups
        self.up = nn.Linear(config.d_model, config.max_ffn_dim * 2, bias=False)
        self.down = nn.Linear(config.max_ffn_dim, config.d_model, bias=False)
        self.group_logits = nn.Parameter(torch.ones(config.ffn_groups) * 1.5)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor, group_gate: torch.Tensor) -> torch.Tensor:
        u, v = self.up(x).chunk(2, dim=-1)
        h = F.silu(u) * v
        g = group_gate.repeat_interleave(self.group_size)
        h = h * g.view(1, 1, -1)
        return self.dropout(self.down(h))


class SOTBlock(nn.Module):
    def __init__(self, config: SOTConfig, index: int):
        super().__init__()
        self.index = index
        self.ln1 = RMSNorm(config.d_model, config.layer_norm_eps)
        self.attn = CausalSelfAttention(config)
        self.ln2 = RMSNorm(config.d_model, config.layer_norm_eps)
        self.ffn = GroupedFFN(config)
        self.layer_logit = nn.Parameter(torch.tensor(config.gate_init_logit))
        if config.skip_span > 0:
            self.skip_logits = nn.Parameter(torch.ones(config.skip_span) * config.gate_init_logit)
        else:
            self.skip_logits = None

    def gate(self) -> torch.Tensor:
        return torch.sigmoid(self.layer_logit)


class SelfOrganizingTransformer(PreTrainedModel):
    config_class = SOTConfig
    base_model_prefix = "sot"
    supports_gradient_checkpointing = True
    _tied_weights_keys = {"lm_head.weight": "embed.weight"}

    def __init__(self, config: SOTConfig):
        super().__init__(config)
        self.embed = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_embed = nn.Embedding(config.max_position_embeddings, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([SOTBlock(config, i) for i in range(config.max_layers)])
        self.ln_f = RMSNorm(config.d_model, config.layer_norm_eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight
        self.post_init()

    def _gates(self):
        if self.config.architecture_mode == "dense":
            device = self.embed.weight.device
            layer = torch.ones(len(self.blocks), device=device)
            heads = torch.ones((len(self.blocks), self.config.max_heads), device=device)
            groups = torch.ones((len(self.blocks), self.config.ffn_groups), device=device)
            skips = torch.zeros((len(self.blocks), self.config.skip_span), device=device) if self.config.skip_span > 0 else torch.empty((len(self.blocks), 0), device=device)
            return layer, heads, groups, skips
        layer = torch.sigmoid(torch.stack([b.layer_logit for b in self.blocks]))
        heads = torch.stack([torch.sigmoid(b.attn.head_logits) for b in self.blocks])
        groups = torch.stack([torch.sigmoid(b.ffn.group_logits) for b in self.blocks])
        if self.config.skip_span > 0:
            skips = torch.stack([torch.sigmoid(b.skip_logits) for b in self.blocks])
        else:
            skips = torch.empty((len(self.blocks), 0), device=layer.device)
        return layer, heads, groups, skips

    def architecture_cost(self) -> torch.Tensor:
        layer, heads, groups, skips = self._gates()
        # Differentiable proxy. Normalised so 1.0 is approximately the dense supernet.
        layer_factor = layer.sum() / self.config.max_layers
        head_factor = (heads.mean(dim=-1) * layer).sum() / self.config.max_layers
        ffn_factor = (groups.mean(dim=-1) * layer).sum() / self.config.max_layers
        skip_factor = skips.mean() if skips.numel() else torch.tensor(0.0, device=layer.device)
        # Attention and FFN dominate, with small topology penalty.
        return 0.45 * head_factor + 0.50 * ffn_factor + 0.05 * layer_factor + 0.02 * skip_factor

    def sparsity_loss(self) -> torch.Tensor:
        layer, heads, groups, skips = self._gates()
        return (
            0.5 * layer.mean()
            + 0.35 * heads.mean()
            + 0.15 * groups.mean()
            + (0.05 * skips.mean() if skips.numel() else 0.0)
        )

    def gate_summary(self) -> dict:
        """Aggregate and per-layer gate means, for logging architecture drift during training."""
        layer, heads, groups, skips = self._gates()
        by_layer = []
        for i in range(len(self.blocks)):
            by_layer.append(
                {
                    "layer": float(layer[i].item()),
                    "head": float(heads[i].mean().item()),
                    "ffn": float(groups[i].mean().item()),
                    "skip": float(skips[i].mean().item()) if skips.numel() else 0.0,
                }
            )
        return {
            "layer_gate_mean": float(layer.mean().item()),
            "head_gate_mean": float(heads.mean().item()),
            "ffn_gate_mean": float(groups.mean().item()),
            "skip_gate_mean": float(skips.mean().item()) if skips.numel() else 0.0,
            "gates_by_layer": by_layer,
        }

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.LongTensor] = None,
        use_cache: bool = False,
        **_: Any,
    ):
        del use_cache
        b, t = input_ids.shape
        if t > self.config.max_position_embeddings:
            raise ValueError(f"Sequence length {t} exceeds max_position_embeddings={self.config.max_position_embeddings}")
        pos = torch.arange(t, device=input_ids.device).unsqueeze(0)
        x = self.drop(self.embed(input_ids) + self.pos_embed(pos))
        previous = []
        layer_gates, head_gates, group_gates, skip_gates = self._gates()

        for i, block in enumerate(self.blocks):
            residual = x
            y = block.ln1(x)
            y = block.attn(y, head_gates[i])
            x = residual + layer_gates[i].view(1, 1, 1) * y
            y = block.ln2(x)
            y = block.ffn(y, group_gates[i])
            x = x + layer_gates[i].view(1, 1, 1) * y

            if i > 0 and self.config.skip_span > 0:
                for s in range(1, min(self.config.skip_span, i) + 1):
                    src = previous[-s]
                    x = x + skip_gates[i, s - 1].view(1, 1, 1) * src / math.sqrt(self.config.skip_span)
            previous.append(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {"loss": loss, "logits": logits}

    def export_architecture(self, threshold: float = 0.5) -> dict:
        layer, heads, groups, skips = self._gates()
        arch = {
            "threshold": threshold,
            "layers": [],
            "d_model": self.config.d_model,
            "max_heads": self.config.max_heads,
            "max_ffn_dim": self.config.max_ffn_dim,
            "ffn_group_size": self.config.max_ffn_dim // self.config.ffn_groups,
        }
        for i in range(self.config.max_layers):
            arch["layers"].append(
                {
                    "index": i,
                    "layer_gate": float(layer[i].item()),
                    "active": bool(layer[i].item() >= threshold),
                    "heads": [int(j) for j, v in enumerate(heads[i].tolist()) if v >= threshold],
                    "ffn_groups": [int(j) for j, v in enumerate(groups[i].tolist()) if v >= threshold],
                    "skip_from": [i - s for s, v in enumerate(skips[i].tolist(), start=1) if v >= threshold and s <= i],
                    "head_gate": heads[i].detach().cpu().tolist(),
                    "ffn_gate": groups[i].detach().cpu().tolist(),
                    "skip_gate": skips[i].detach().cpu().tolist(),
                }
            )
        return arch


__all__ = ["SOTConfig", "SelfOrganizingTransformer"]


In [ ]:
%%writefile train.py
from __future__ import annotations

import argparse
import json
import math
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List

import numpy as np
import torch
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo
from tqdm import tqdm
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup, set_seed

from model import SOTConfig, SelfOrganizingTransformer


@dataclass
class TrainConfig:
    model_type: str = "sot"
    tokenizer_name: str = "gpt2"
    dataset_name: str = "roneneldan/TinyStories"
    dataset_config: str | None = None
    text_column: str = "text"
    max_train_docs: int | None = None
    seq_len: int = 256
    vocab_size: int = 50257
    max_layers: int = 12
    d_model: int = 384
    max_heads: int = 6
    max_ffn_dim: int = 1536
    ffn_groups: int = 64
    skip_span: int = 3
    dropout: float = 0.0
    batch_size: int = 16
    grad_accum: int = 8
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 500
    max_steps: int = 10000
    eval_every: int = 500
    save_every: int = 1000
    log_every: int = 25
    grad_clip: float = 1.0
    arch_reg: float = 0.015
    target_cost: float = 0.45
    lagrangian_lr: float = 0.01
    gate_warmup_steps: int = 1000
    gate_init_logit: float = -1.0
    device: str = "auto"
    seed: int = 42
    num_workers: int = 2
    output_dir: str = "./outputs/sot"
    push_to_hub: bool = False
    hub_model_id: str | None = None
    hub_private: bool = False


class PackedTokens:
    def __init__(self, token_ids: List[int], seq_len: int):
        self.tokens = np.asarray(token_ids, dtype=np.int32)
        self.seq_len = seq_len
        self.n = (len(self.tokens) - 1) // seq_len

    def __len__(self):
        return max(0, self.n)

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = self.tokens[start : start + self.seq_len]
        return torch.from_numpy(x.astype(np.int64))


class BatchIterator:
    def __init__(self, packed: PackedTokens, batch_size: int, seed: int = 42):
        self.packed = packed
        self.batch_size = batch_size
        self.rng = random.Random(seed)

    def __iter__(self):
        order = list(range(len(self.packed)))
        self.rng.shuffle(order)
        for i in range(0, len(order), self.batch_size):
            ids = order[i : i + self.batch_size]
            if len(ids) < self.batch_size:
                continue
            yield torch.stack([self.packed[j] for j in ids])


def pick_device(name: str) -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def tokenise_split(ds, tokenizer, text_column: str, max_docs: int | None = None) -> List[int]:
    if max_docs is not None and max_docs < len(ds):
        # Tokenizing is a single-threaded Python loop over the whole split; for a short
        # diagnostic run that only needs a few million tokens, capping the document count
        # avoids paying for a full pass over a multi-million-row corpus every invocation.
        ds = ds.select(range(max_docs))
    out: List[int] = []
    eos = tokenizer.eos_token_id
    for row in tqdm(ds, desc="tokenizing", unit="doc"):
        text = row[text_column]
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        out.extend(ids)
        if eos is not None:
            out.append(eos)
    return out


def load_tokens(cfg: TrainConfig, tokenizer):
    kwargs = {}
    if cfg.dataset_config:
        raw = load_dataset(cfg.dataset_name, cfg.dataset_config)
    else:
        raw = load_dataset(cfg.dataset_name)
    train_tokens = tokenise_split(raw["train"], tokenizer, cfg.text_column, cfg.max_train_docs)
    val_split = "validation" if "validation" in raw else "train"
    if val_split == "train":
        # Deterministic small holdout from the tail if no validation split exists.
        cut = int(len(train_tokens) * 0.99)
        val_tokens = train_tokens[cut:]
        train_tokens = train_tokens[:cut]
    else:
        val_tokens = tokenise_split(raw[val_split], tokenizer, cfg.text_column)
    return PackedTokens(train_tokens, cfg.seq_len), PackedTokens(val_tokens, cfg.seq_len)


def build_model(cfg: TrainConfig, tokenizer) -> SelfOrganizingTransformer:
    config = SOTConfig(
        vocab_size=len(tokenizer),
        max_position_embeddings=cfg.seq_len,
        max_layers=cfg.max_layers,
        d_model=cfg.d_model,
        max_heads=cfg.max_heads,
        max_ffn_dim=cfg.max_ffn_dim,
        ffn_groups=cfg.ffn_groups,
        skip_span=cfg.skip_span,
        dropout=cfg.dropout,
        architecture_mode=cfg.model_type,
        gate_init_logit=cfg.gate_init_logit,
    )
    model = SelfOrganizingTransformer(config)
    return model


def evaluate_loss(model, packed: PackedTokens, batch_size: int, device: torch.device, max_batches: int = 50) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for bi, batch in enumerate(BatchIterator(packed, batch_size, seed=1234)):
            batch = batch.to(device)
            out = model(batch, labels=batch)
            losses.append(float(out["loss"].item()))
            if bi + 1 >= max_batches:
                break
    model.train()
    return float(np.mean(losses)) if losses else float("inf")


def save_checkpoint(model, tokenizer, cfg: TrainConfig, out_dir: Path, step: int, history: list, is_final=False):
    path = out_dir / ("final" if is_final else f"checkpoint-{step}")
    path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(path, safe_serialization=True)
    tokenizer.save_pretrained(path)
    with open(path / "train_config.json", "w") as f:
        json.dump(asdict(cfg), f, indent=2)
    with open(path / "architecture.json", "w") as f:
        json.dump(model.export_architecture(), f, indent=2)
    with open(path / "metrics.json", "w") as f:
        json.dump(history, f, indent=2)
    return path


def push_folder(path: Path, repo_id: str, private: bool):
    api = HfApi()
    create_repo(repo_id=repo_id, private=private, exist_ok=True)
    api.upload_folder(folder_path=str(path), repo_id=repo_id, repo_type="model")
    print(f"Uploaded {path} -> https://huggingface.co/{repo_id}")


def train(cfg: TrainConfig):
    set_seed(cfg.seed)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)
    torch.manual_seed(cfg.seed)
    device = pick_device(cfg.device)
    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg.tokenizer_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    train_data, val_data = load_tokens(cfg, tokenizer)

    model = build_model(cfg, tokenizer).to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay, betas=(0.9, 0.95))
    scheduler = get_cosine_schedule_with_warmup(optimizer, cfg.warmup_steps, cfg.max_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    # Learned Lagrange multiplier for the compute budget.
    lagrange = torch.tensor(0.0, device=device)
    history = []
    global_step = 0
    micro_step = 0
    train_iter = iter(BatchIterator(train_data, cfg.batch_size, seed=cfg.seed))

    pbar = tqdm(total=cfg.max_steps, desc="train", unit="step")
    while global_step < cfg.max_steps:
        optimizer.zero_grad(set_to_none=True)
        accum_task = 0.0
        accum_cost = 0.0
        for _ in range(cfg.grad_accum):
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(BatchIterator(train_data, cfg.batch_size, seed=cfg.seed + global_step + 1))
                batch = next(train_iter)
            batch = batch.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16, enabled=(device.type in {"cuda", "cpu"})):
                out = model(batch, labels=batch)
                task_loss = out["loss"]
                cost = model.architecture_cost()
                # Warm up architectural pressure so useful representations form before pruning pressure.
                progress = min(1.0, global_step / max(1, cfg.gate_warmup_steps))
                reg = cfg.arch_reg * progress * model.sparsity_loss()
                constrained = lagrange.detach() * (cost - cfg.target_cost)
                loss = (task_loss + reg + constrained) / cfg.grad_accum
            scaler.scale(loss).backward()
            accum_task += float(task_loss.detach())
            accum_cost += float(cost.detach())
            micro_step += 1

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        mean_cost = accum_cost / cfg.grad_accum
        # Projected dual ascent. Restricting lambda prevents runaway regularisation.
        with torch.no_grad():
            lagrange += cfg.lagrangian_lr * (mean_cost - cfg.target_cost)
            lagrange.clamp_(0.0, 10.0)
        global_step += 1
        train_loss = accum_task / cfg.grad_accum
        pbar.update(1)
        pbar.set_postfix(loss=f"{train_loss:.3f}", cost=f"{mean_cost:.3f}", lgr=f"{float(lagrange.item()):.3f}")

        if global_step % cfg.log_every == 0 or global_step == 1:
            record = {
                "step": global_step,
                "train_loss": train_loss,
                "cost": mean_cost,
                "lagrange": float(lagrange.item()),
                "lr": scheduler.get_last_lr()[0],
                **model.gate_summary(),
            }
            pbar.write(json.dumps(record))
            history.append(record)

        if global_step % cfg.eval_every == 0 or global_step == cfg.max_steps:
            val_loss = evaluate_loss(model, val_data, cfg.batch_size, device)
            record = {"step": global_step, "val_loss": val_loss, "val_ppl": math.exp(min(20, val_loss))}
            pbar.write(json.dumps(record))
            history.append(record)
            save_checkpoint(model, tokenizer, cfg, out_dir, global_step, history)

        if global_step % cfg.save_every == 0:
            save_checkpoint(model, tokenizer, cfg, out_dir, global_step, history)

    pbar.close()
    final_path = save_checkpoint(model, tokenizer, cfg, out_dir, global_step, history, is_final=True)
    print("\nDISCOVERED ARCHITECTURE")
    print(json.dumps(model.export_architecture(threshold=0.5), indent=2))
    if cfg.push_to_hub:
        if not cfg.hub_model_id:
            raise ValueError("--hub_model_id is required with --push_to_hub")
        push_folder(final_path, cfg.hub_model_id, cfg.hub_private)


def parse_args() -> TrainConfig:
    p = argparse.ArgumentParser()
    p.add_argument("--model_type", choices=["sot", "dense"], default="sot")
    p.add_argument("--output_dir", default="./outputs/sot")
    p.add_argument("--dataset_name", default="roneneldan/TinyStories")
    p.add_argument("--max_train_docs", type=int, default=None, help="Cap on train-split documents to tokenize, for fast diagnostic runs")
    p.add_argument("--tokenizer_name", default="gpt2")
    p.add_argument("--seq_len", type=int, default=256)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--grad_accum", type=int, default=8)
    p.add_argument("--max_steps", type=int, default=10000)
    p.add_argument("--eval_every", type=int, default=500)
    p.add_argument("--save_every", type=int, default=1000)
    p.add_argument("--learning_rate", type=float, default=3e-4)
    p.add_argument("--arch_reg", type=float, default=0.015)
    p.add_argument("--target_cost", type=float, default=0.45)
    p.add_argument("--lagrangian_lr", type=float, default=0.01)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--device", default="auto")
    p.add_argument("--max_layers", type=int, default=12)
    p.add_argument("--d_model", type=int, default=384)
    p.add_argument("--max_heads", type=int, default=6)
    p.add_argument("--max_ffn_dim", type=int, default=1536)
    p.add_argument("--ffn_groups", type=int, default=64)
    p.add_argument("--skip_span", type=int, default=3)
    p.add_argument("--gate_init_logit", type=float, default=-1.0)
    p.add_argument("--push_to_hub", action="store_true")
    p.add_argument("--hub_model_id", default=None)
    p.add_argument("--hub_private", action="store_true")
    a = p.parse_args()
    return TrainConfig(**vars(a))


if __name__ == "__main__":
    train(parse_args())


## Minimal baseline: run the diagnostic

2000 steps at the default effective batch (16 x grad_accum 8 = 128 sequences of 256 tokens)
is ~65.5M tokens. `--max_train_docs 400000` caps tokenization to roughly that much data
(TinyStories docs run ~150-200 tokens each here) instead of the full 2.1M-row corpus — still
well under 20% of it, so it's not repeating the same slice much. On a T4 this should take
well under an hour; there's no checkpoint-resume in `train.py`, so if you want to sanity-check
the pipeline first, drop `--max_steps` for a quick pass rather than interrupting this one and
expecting to continue it later — a lower step count is a separate, shorter run, not a
prefix of this one.


In [ ]:
!python train.py --model_type sot --output_dir outputs/diag \
  --max_train_docs 400000 \
  --max_steps 2000 --eval_every 200 --save_every 2000 --log_every 20


## Results

Load the logged history and plot:
1. `cost` over training against the `target_cost=0.45` line — does it ever approach the
   budget, or does it just fall further below its 0.16 starting point?
2. Per-family gate means (layer/head/ffn/skip) over training — do the families move at
   different rates, or all shrink together?
3. A per-layer heatmap at the final logged step — is there depth-dependent structure, or is
   every layer doing the same thing?


In [ ]:
import json
import matplotlib.pyplot as plt

with open("outputs/diag/final/metrics.json") as f:
    history = json.load(f)

# Two kinds of record get appended to history: per-log_every gate/cost records (have
# "gates_by_layer"), and per-eval_every val-loss records (have "val_loss"). Split them.
train_records = [r for r in history if "gates_by_layer" in r]
val_records = [r for r in history if "val_loss" in r]

print(f"{len(train_records)} gate-logging records, {len(val_records)} eval records")
train_records[0]


In [ ]:
steps = [r["step"] for r in train_records]
target_cost = 0.45  # matches train.py's --target_cost default used above

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(steps, [r["cost"] for r in train_records], label="cost")
axes[0].axhline(target_cost, color="red", linestyle="--", label="target_cost")
axes[0].set_xlabel("step"); axes[0].set_ylabel("architecture_cost()")
axes[0].set_title("Cost vs budget"); axes[0].legend()

for family in ["layer_gate_mean", "head_gate_mean", "ffn_gate_mean", "skip_gate_mean"]:
    axes[1].plot(steps, [r[family] for r in train_records], label=family.replace("_gate_mean", ""))
axes[1].set_xlabel("step"); axes[1].set_ylabel("mean gate value")
axes[1].set_title("Per-family gate means"); axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

last = train_records[-1]
families = ["layer", "head", "ffn", "skip"]
grid = np.array([[layer[f] for f in families] for layer in last["gates_by_layer"]])

fig, ax = plt.subplots(figsize=(4, 6))
im = ax.imshow(grid, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax.set_xticks(range(len(families))); ax.set_xticklabels(families)
ax.set_yticks(range(len(last["gates_by_layer"])))
ax.set_ylabel("layer (depth)")
ax.set_title(f"Gate means by layer at step {last['step']}")
fig.colorbar(im, label="mean gate value")
plt.tight_layout()
plt.show()


In [ ]:
# Quantify movement directly: first logged step vs last, per family.
# A near-zero delta across the board means the regularizer/task-loss balance isn't doing
# anything interesting yet at this step count -- run longer or revisit target_cost/gate_init.
first = train_records[0]
last = train_records[-1]
print(f"{'family':<10}{'step ' + str(first['step']):>12}{'step ' + str(last['step']):>12}{'delta':>10}")
for family in ["layer_gate_mean", "head_gate_mean", "ffn_gate_mean", "skip_gate_mean"]:
    delta = last[family] - first[family]
    print(f"{family.replace('_gate_mean',''):<10}{first[family]:>12.4f}{last[family]:>12.4f}{delta:>10.4f}")

print(f"\ncost: {first['cost']:.4f} -> {last['cost']:.4f} (target_cost={target_cost})")
print(f"lagrange: {first['lagrange']:.4f} -> {last['lagrange']:.4f}",
      "(stays 0 for the whole run == the budget constraint never activated)")


- Key observations:
- Surprises or failure modes:
- Decision: continue, pivot, or stop:

## Next steps

- If gates moved and diverged by layer: this run's `metrics.json` and
  `outputs/diag/final/architecture.json` are enough evidence to decide `target_cost` /
  `gate_init_logit` / `arch_reg` for a real Phase-1 run — no need to guess.
- If gates barely moved or moved uniformly even by step 2000 (well past both the LR and
  `arch_reg` warmups): that's real evidence the mechanism is inert at these defaults, not an
  artifact of stopping too early. Try `--max_steps 4000`+ only to double-check before
  concluding that, since there's no resume — each longer check is a fresh full run.
- If `cost` never approaches `target_cost`: that confirms the Lagrangian term is inert at
  these defaults; the next real decision is whether to lower `target_cost`, raise
  `gate_init_logit`, or accept that task-loss vs `sparsity_loss` tension alone is sufficient
  and the Lagrangian term is just acting as an unused safety ceiling.
- To download the checkpoint/architecture for further analysis: `outputs/diag/final/` in the
  Colab file browser, or zip and download it with `!zip -r diag.zip outputs/diag/final`.
